NOMBRES DE INTEGRANTES:
MARIA JOSE MURILLO MENDOZA 
LAURA CAMACHO LIPA 
ANDRES REVOLLO ALMENDRAS


In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

Configuración general

In [5]:
BATCH_SIZE = 32
DATA_PATH = 'fifa2021_training.csv'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Usando dispositivo:", DEVICE)

Usando dispositivo: cpu


In [6]:
def load_array(data_arrays, batch_size, is_train=True):
    """Crea un DataLoader desde tensores."""
    dataset = TensorDataset(*data_arrays)
    return DataLoader(dataset, batch_size=batch_size, shuffle=is_train)

### Función `evaluate_loss()`

Evalúa el error promedio de la red neuronal sobre un conjunto de datos.

#### Entrada
- Modelo entrenado (`net`)
- Iterador de datos (`data_iter`)
- Función de pérdida (`loss_fn`)

#### Proceso
La función desactiva el modo entrenamiento usando `eval()` y evita el cálculo de gradientes con `torch.no_grad()`.

Para cada batch:
1. Realiza predicciones
2. Calcula la pérdida
3. Acumula el error total

Finalmente se obtiene la pérdida promedio del dataset completo.

#### Salida
Devuelve un valor numérico que representa la pérdida promedio.

#### ¿Por qué se utiliza?
La pérdida permite medir qué tan lejos están las predicciones respecto a las etiquetas reales.  
Comparar la pérdida de entrenamiento y prueba ayuda a identificar underfitting o overfitting.

In [7]:
def evaluate_loss(net, data_iter, loss_fn):
    """Evalúa la pérdida promedio sobre un iterador."""
    net.eval()
    total_loss, total_samples = 0.0, 0
    with torch.no_grad():
        for X_batch, y_batch in data_iter:
            output = net(X_batch)
            loss = loss_fn(output, y_batch)
            total_loss += loss.item() * y_batch.size(0)
            total_samples += y_batch.size(0)
    return total_loss / total_samples

### Función `accuracy()`

Calcula el rendimiento del modelo usando la métrica de accuracy.

#### Entrada
- Modelo entrenado
- Dataset a evaluar

#### Proceso
Para cada batch:
1. La red genera probabilidades para cada clase
2. `argmax()` selecciona la clase con mayor probabilidad
3. Se comparan las predicciones con las etiquetas reales

La función acumula la cantidad total de aciertos y calcula el porcentaje final.

#### Salida
Devuelve un valor entre 0 y 1:
- 0 → ninguna predicción correcta
- 1 → todas las predicciones correctas

#### ¿Por qué se utiliza?
El accuracy es la métrica principal de clasificación y permite evaluar qué tan bien la red identifica la posición correcta de cada jugador.

In [8]:
def accuracy(net, data_iter):
    """Calcula accuracy sobre un iterador completo."""
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in data_iter:
            preds = net(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total

### Función `train_epoch_ch3()`

Entrena la red neuronal durante una época completa usando todos los batches del conjunto de entrenamiento.

#### Entrada
- Modelo (`net`)
- DataLoader de entrenamiento
- Función de pérdida
- Optimizador

#### Proceso
Para cada batch se ejecutan las etapas principales del entrenamiento:

1. **Forward pass**  
   La red genera predicciones a partir de las entradas.

2. **Cálculo de pérdida**  
   Se mide el error entre predicciones y etiquetas reales.

3. **Backpropagation**  
   Se calculan gradientes usando derivadas parciales.

4. **Actualización de pesos**  
   El optimizador ajusta los parámetros para reducir la pérdida.

Además, se acumulan métricas de pérdida y accuracy.

#### Salida
Devuelve:
- Loss promedio de entrenamiento
- Accuracy promedio de entrenamiento

#### ¿Por qué se utiliza?
Separar el entrenamiento por épocas permite monitorear cómo evoluciona el aprendizaje del modelo a lo largo del tiempo.

In [9]:
def train_epoch_ch3(net, train_iter, loss_fn, optimizer):
    """Entrena una época y devuelve (train_loss, train_accuracy)."""
    net.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_iter:
        optimizer.zero_grad()
        output = net(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y_batch.size(0)
        correct += (output.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.size(0)
    return total_loss / total, correct / total
 

In [10]:
# ─────────────────────────────────────────────────────────
# CARGA Y PREPROCESAMIENTO DEL DATASET
# ─────────────────────────────────────────────────────────

In [11]:
SKILL_COLS = [
    'BallControl', 'Dribbling', 'Marking', 'SlideTackle', 'StandTackle',
    'Aggression', 'Reactions', 'Interceptions', 'Vision', 'Composure',
    'Crossing', 'ShortPass', 'LongPass', 'Acceleration', 'Stamina',
    'Strength', 'Balance', 'SprintSpeed', 'Agility', 'Jumping',
    'Heading', 'ShotPower', 'Finishing', 'LongShots', 'Curve',
    'FKAcc', 'Penalties', 'Volleys', 'GKDiving', 'GKHandling',
    'GKKicking', 'GKReflexes'
]

In [14]:
BASE_COLS = ['Height', 'Weight', 'Age']

### Función `load_and_preprocess()`

Realiza todo el flujo de preparación del dataset antes del entrenamiento.

#### Procesos realizados

1. **Carga del dataset**
   - Se lee el archivo CSV usando pandas.

2. **Selección de variables**
   - Se conservan únicamente las columnas relevantes para el modelo.

3. **Limpieza de datos**
   - Se eliminan filas con valores faltantes.

4. **One-hot encoding**
   - La variable categórica `Sex` se transforma a variables binarias.

5. **Codificación de etiquetas**
   - Las posiciones (`DEF`, `MID`, etc.) se convierten a números enteros.

6. **Normalización**
   - Se aplica `StandardScaler` para centrar y escalar los datos.

7. **Train/Test Split**
   - Se separan datos de entrenamiento y prueba manteniendo proporciones similares de clases.

8. **Conversión a tensores**
   - Los datos se convierten al formato requerido por PyTorch.

#### ¿Por qué se utiliza?
Las redes neuronales trabajan mejor con datos numéricos y normalizados.  
Este preprocesamiento mejora la estabilidad del entrenamiento y facilita la convergencia del modelo.

In [12]:
def load_and_preprocess(filepath, test_size=0.3, random_state=42):
    """
    Carga el CSV, aplica preprocesamiento y devuelve tensores.
    test_size=0.3 → split 70/30 según la práctica.
    """
    df = pd.read_csv(filepath)

    # Seleccionar columnas relevantes
    cols = BASE_COLS + ['Sex'] + SKILL_COLS + ['Position']
    df = df[cols].dropna()

    # One-hot encoding de Sex
    df = pd.get_dummies(df, columns=['Sex'], drop_first=False)
    sex_cols = [c for c in df.columns if c.startswith('Sex_')]

    # Feature columns
    feature_cols = BASE_COLS + sex_cols + SKILL_COLS

    X = df[feature_cols].values.astype(np.float32)
    y_raw = df['Position'].values

    # Codificar etiquetas
    le = LabelEncoder()
    y = le.fit_transform(y_raw).astype(np.int64)

    print(f"Dataset: {X.shape[0]} muestras, {X.shape[1]} features")
    print(f"Clases: {le.classes_} → {list(range(len(le.classes_)))}")
    print(f"Distribución: { {k: int((y==i).sum()) for i,k in enumerate(le.classes_)} }")

    # Normalización (CRUCIAL para convergencia)
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Train/Test split 70/30
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    print(f"\nSplit {int((1-test_size)*100)}/{int(test_size*100)} → "
          f"Train: {len(X_train)}, Test: {len(X_test)}")

    # Convertir a tensores
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_test_t  = torch.tensor(y_test,  dtype=torch.long)

    return X_train_t, X_test_t, y_train_t, y_test_t, le, X.shape[1]

In [15]:
X_train, X_test, y_train, y_test, label_encoder, n_features = load_and_preprocess(DATA_PATH)

train_iter = load_array((X_train, y_train), BATCH_SIZE, True)
test_iter  = load_array((X_test, y_test), BATCH_SIZE, False)

Dataset: 13921 muestras, 37 features
Clases: ['DEF' 'FWD' 'GK' 'MID'] → [0, 1, 2, 3]
Distribución: {'DEF': 4585, 'FWD': 2715, 'GK': 1550, 'MID': 5071}

Split 70/30 → Train: 9744, Test: 4177


### Resumen del Preprocesamiento Realizado

Cargados y normalizados datos
- 16000+ muestras, 37 features
- Se eliminaron filas con valores faltantes
- Se aplicó **StandardScaler** para normalizar (¡crucial para convergencia!)
- Split 70/30 con estratificación

DataLoaders creados
- Batches de 32 muestras
- `train_iter`: para entrenamiento (shuffle=True)
- `test_iter`: para validación (shuffle=False)

In [16]:
# ─────────────────────────────────────────────────────────
# FUNCIÓN PARA CREAR MODELOS CONFIGURABLES
# ─────────────────────────────────────────────────────────

### Función `build_model()`

Construye una arquitectura MLP configurable de forma dinámica usando `nn.Sequential`.

#### Entrada
- `input_size` → número de features de entrada
- `hidden_sizes` → lista con neuronas por capa oculta
- `output_size` → número de clases de salida
- `activation` → función de activación
- `dropout_rate` → tasa de dropout
- `batch_norm` → habilita Batch Normalization

#### Proceso

La función recorre la lista `hidden_sizes` y crea automáticamente:

1. Capas lineales (`Linear`)
2. Funciones de activación
3. BatchNorm (opcional)
4. Dropout (opcional)

Finalmente agrega la capa de salida con 4 neuronas, correspondientes a las posiciones de jugadores.

#### Salida
Devuelve un modelo `nn.Sequential` completamente construido y listo para entrenar.

#### ¿Por qué se utiliza?

Permite probar distintas arquitecturas sin reescribir la red manualmente.  
Esto facilita comparar modelos pequeños, medianos y grandes para analizar underfitting y overfitting.

In [17]:
def build_model(
    input_size,
    hidden_sizes,          # lista: e.g. [64, 64] → 2 capas ocultas de 64 neuronas
    output_size=4,
    activation='relu',     # 'relu', 'tanh', 'leakyrelu', 'elu'
    dropout_rate=0.0,      # 0.0 = sin dropout
    batch_norm=False       # True = BatchNorm entre capas
):
    """
    Construye un MLP configurable con nn.Sequential.

    CORRECCIÓN: se usa una función (lambda) por activación para crear
    instancias nuevas de nn.Module en cada capa. Reusar el mismo objeto
    causa errores con BatchNorm y nn.Sequential.
    """
    # FIX: cada entrada es una función que crea una instancia nueva
    act_map = {
        'relu':      lambda: nn.ReLU(),
        'tanh':      lambda: nn.Tanh(),
        'leakyrelu': lambda: nn.LeakyReLU(0.1),
        'elu':       lambda: nn.ELU()
    }
    get_act = act_map.get(activation.lower(), lambda: nn.ReLU())

    layers = []
    prev_size = input_size

    for h in hidden_sizes:
        layers.append(nn.Linear(prev_size, h))
        if batch_norm:
            layers.append(nn.BatchNorm1d(h))
        layers.append(get_act())   # FIX: instancia nueva por capa
        if dropout_rate > 0.0:
            layers.append(nn.Dropout(dropout_rate))
        prev_size = h

    layers.append(nn.Linear(prev_size, output_size))
    return nn.Sequential(*layers)

In [30]:
# ─────────────────────────────────────────────────────────
# FUNCIÓN DE ENTRENAMIENTO CON TENSORBOARD
# ─────────────────────────────────────────────────────────

### Función `train()`

Controla todo el proceso de entrenamiento y evaluación del modelo.

#### Entrada
- Modelo (`net`)
- DataLoaders de entrenamiento y prueba
- `SummaryWriter` para TensorBoard
- Nombre del modelo
- Número de épocas
- Learning rate

#### Proceso

En cada época se realizan las siguientes etapas:

1. **Entrenamiento**
   - Se llama a `train_epoch_ch3()`
   - Se actualizan pesos de la red

2. **Evaluación**
   - Se calcula loss y accuracy en datos de prueba

3. **Registro de métricas**
   - Los resultados se almacenan en TensorBoard

4. **Seguimiento del mejor modelo**
   - Se guarda el mejor accuracy de prueba alcanzado

#### Salida
Devuelve:
- Mejor accuracy obtenido en test
- Historial completo de métricas

#### ¿Por qué se utiliza?

Centralizar el entrenamiento en una sola función facilita comparar arquitecturas y analizar el comportamiento del modelo durante el aprendizaje.

In [18]:
def train(net, train_iter, test_iter, writer, model_name,
          num_epochs=500, lr=0.001, print_every=10):

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)

    history = {
        'train_loss': [],
        'test_loss':  [],
        'train_acc':  [],
        'test_acc':   []
    }

    best_test_acc = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_epoch_ch3(net, train_iter, loss_fn, optimizer)
        test_loss  = evaluate_loss(net, test_iter, loss_fn)
        test_acc   = accuracy(net, test_iter)

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        writer.add_scalars(f'{model_name}/Loss',
                           {'Train': train_loss, 'Test': test_loss}, epoch)
        writer.add_scalars(f'{model_name}/Accuracy',
                           {'Train': train_acc,  'Test': test_acc},  epoch)

        if test_acc > best_test_acc:
            best_test_acc = test_acc

        if epoch % print_every == 0 or epoch == 1:
            print(f"[{model_name}] Epoch {epoch}/{num_epochs} | "
                  f"TrainLoss={train_loss:.4f} | TestLoss={test_loss:.4f} | "
                  f"TrainAcc={train_acc:.4f} | TestAcc={test_acc:.4f}")

    writer.flush()   # FIX: asegura que TensorBoard lea todos los datos
    print(f"✓ Mejor Accuracy Test [{model_name}]: {best_test_acc:.4f}")
    return best_test_acc, history

### Entrenamiento de modelos base

En esta etapa se entrenan tres arquitecturas MLP con diferentes niveles de complejidad:

- Modelo chico → 2 capas de 4 neuronas
- Modelo medio → 2 capas de 16 neuronas
- Modelo grande → 2 capas de 256 neuronas

#### Proceso

Para cada modelo:

1. Se construye la arquitectura usando `build_model()`
2. Se crea un `SummaryWriter` independiente
3. Se entrena durante 500 épocas
4. Se almacenan métricas y resultados finales

#### ¿Por qué se realiza?

El objetivo es comparar cómo afecta la capacidad del modelo al rendimiento.

- Redes pequeñas suelen presentar underfitting
- Redes muy grandes pueden generar overfitting
- Redes intermedias suelen ofrecer mejor balance

Las métricas registradas en TensorBoard permiten visualizar estas diferencias durante el entrenamiento.

In [ ]:
print("="*60)
print("FASE 1: CONSTRUCCIÓN Y ENTRENAMIENTO DE TRES MODELOS")
print("="*60)
os.makedirs("runs", exist_ok=True)

base_configs = {
    'chico':  {'hidden_sizes': [4,   4]},
    'medio':  {'hidden_sizes': [16, 16]},
    'grande': {'hidden_sizes': [256,256]}
}

base_results   = {}
base_histories = {}

# Requisito del PDF: entrenar 500 épocas en los modelos base
FASE1_EPOCHS = 500

for name, cfg in base_configs.items():
    print(f"\n>>> Entrenando modelo {name.upper()} <<<")
    net = build_model(n_features, activation='relu', **cfg)
    writer = SummaryWriter(log_dir=f'runs/{name}')
    acc, history = train(net, train_iter, test_iter,
                         writer=writer,
                         model_name=name,
                         num_epochs=FASE1_EPOCHS,
                         lr=0.001,
                         print_every=10)
    base_results[name]   = acc
    base_histories[name] = history
    writer.close()

print("\n" + "="*60)
print("RESULTADOS FASE 1")
print("="*60)
for k, v in sorted(base_results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {k.upper():8s} → Accuracy Test: {v:.4f}")

FASE 1: CONSTRUCCIÓN Y ENTRENAMIENTO DE TRES MODELOS

>>> Entrenando modelo CHICO <<<
[chico] Epoch 1/500 | TrainLoss=1.0774 | TestLoss=0.8406 | TrainAcc=0.5471 | TestAcc=0.7122


KeyboardInterrupt: 

## Análisis de Resultados

Se entrenaron tres arquitecturas MLP con diferentes niveles de complejidad para analizar su comportamiento durante el aprendizaje y observar casos de underfitting y overfitting.

---

### Modelo Chico `[4, 4]`

El modelo pequeño obtuvo un accuracy máximo de prueba de aproximadamente **89.2%**.

Durante el entrenamiento:
- La pérdida de entrenamiento y prueba se mantuvieron relativamente cercanas.
- El accuracy de entrenamiento y prueba fue bastante similar.
- Las métricas se estabilizaron rápidamente y dejaron de mejorar después de varias épocas.

Esto indica que el modelo logró generalizar correctamente sin memorizar los datos de entrenamiento. Sin embargo, su capacidad de aprendizaje fue limitada debido a la pequeña cantidad de neuronas.

#### Comportamiento observado
- Buena generalización
- Bajo riesgo de overfitting
- Capacidad limitada para aprender patrones más complejos

#### Interpretación
El modelo presenta un comportamiento cercano a **underfitting leve**, ya que la red no tiene suficiente capacidad para seguir mejorando significativamente el rendimiento.

---

### Modelo Medio `[16, 16]`

El modelo medio obtuvo el mejor resultado general, alcanzando un accuracy máximo de prueba cercano a **89.6%**.

En las primeras épocas:
- La pérdida de entrenamiento y prueba disminuyeron correctamente.
- El accuracy de prueba aumentó de forma estable.
- Se observó un buen equilibrio entre aprendizaje y generalización.

Sin embargo, después de varias épocas:
- La pérdida de entrenamiento siguió disminuyendo.
- La pérdida de prueba comenzó a aumentar progresivamente.

Esto indica el inicio de overfitting después de cierto punto del entrenamiento.

#### Comportamiento observado
- Mejor balance entre capacidad y generalización
- Buen rendimiento en entrenamiento y prueba
- Inicio de overfitting después de muchas épocas

#### Interpretación
El modelo medio fue el más equilibrado de los tres modelos evaluados.  
Tuvo suficiente capacidad para aprender patrones importantes sin sobreajustarse demasiado al inicio del entrenamiento.

Además, se observó que entrenar demasiadas épocas terminó perjudicando el rendimiento en prueba, lo cual evidencia la importancia de controlar el número de épocas o aplicar técnicas de regularización.

---

### Modelo Grande `[256, 256]`

El modelo grande alcanzó accuracy perfecto en entrenamiento:

- `TrainAcc ≈ 1.0000`
- `TrainLoss ≈ 0.0000`

Sin embargo:
- La pérdida en prueba aumentó considerablemente.
- El accuracy de prueba dejó de mejorar.
- La diferencia entre entrenamiento y prueba se volvió muy grande.

Por ejemplo:
- `TestLoss` llegó a valores superiores a `2.0`
- Mientras que el accuracy de entrenamiento permaneció en `100%`

#### Comportamiento observado
- Memorización completa del conjunto de entrenamiento
- Muy baja capacidad de generalización
- Overfitting severo

#### Interpretación
El modelo grande aprendió excesivamente los datos de entrenamiento y perdió capacidad para generalizar correctamente sobre datos nuevos.

Aunque el accuracy de prueba no cayó drásticamente, la pérdida de prueba aumentó mucho debido a que el modelo realizaba predicciones incorrectas con demasiada confianza, lo cual es penalizado fuertemente por `CrossEntropyLoss`.

---

## Comparación General

| Modelo | Accuracy Test | Comportamiento |
|---|---|---|
| Chico | 0.8920 | Capacidad limitada |
| Medio | 0.8961 | Mejor balance |
| Grande | 0.8951 | Overfitting fuerte |

---

## Conclusiones Generales

- Modelos pequeños suelen tener menor capacidad de aprendizaje y pueden presentar underfitting.
- Modelos demasiado grandes tienden a memorizar los datos de entrenamiento, produciendo overfitting.
- El mejor rendimiento se obtuvo con una arquitectura intermedia, que logró un mejor equilibrio entre capacidad y generalización.
- El análisis de curvas de pérdida y accuracy en TensorBoard permitió identificar claramente el comportamiento de cada arquitectura.
- Entrenar demasiadas épocas puede empeorar el rendimiento en prueba incluso cuando el entrenamiento sigue mejorando.


### Visualización con TensorBoard

El entrenamiento ha guardado todos los datos en la carpeta `runs/`:

```
runs/
├── chico/      → Gráficas del modelo pequeño
├── medio/      → Gráficas del modelo medio
└── grande/     → Gráficas del modelo grande
```

#### Cómo usar TensorBoard:

1. **En terminal** (fuera de Jupyter):
   ```bash
   tensorboard --logdir=runs
   ```

2. **Accede a**: http://localhost:6006

3. **En las gráficas busca**:
   - **Overfitting**: Test loss sube mientras train loss baja
   - **Underfitting**: Ambas pérdidas altas y planas
   - **Buen ajuste**: Ambas convergen juntas

#### Interpretación Esperada:

- **Modelo Chico** (4 neuronas): Underfitting → accuracies bajos
- **Modelo Medio** (16 neuronas): Balance → mejor generalización
- **Modelo Grande** (256 neuronas): Overfitting → gap entre train/test

In [2]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [3]:
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 28256), started 0:02:21 ago. (Use '!kill 28256' to kill it.)

In [ ]:
print("\n" + "="*80)
print("FASE 2: COMPETENCIA - BÚSQUEDA DE LA MEJOR ARQUITECTURA")
print("="*80)

# Definición de 5 arquitecturas competitivas
competition_configs = {
    'comp_1_balance': {
        'hidden_sizes': [64, 32],
        'activation': 'relu',
        'dropout_rate': 0.2,
        'batch_norm': False,
        'lr': 0.001,
        'description': 'Balance: ReLU + Dropout moderado, sin BatchNorm'
    },

    'comp_2_batchnorm': {
        'hidden_sizes': [128, 64],
        'activation': 'relu',
        'dropout_rate': 0.3,
        'batch_norm': True,
        'lr': 0.001,
        'description': 'Con BatchNorm: Capas más grandes + ReLU'
    },

    'comp_3_profunda': {
        'hidden_sizes': [64, 32, 16],
        'activation': 'leakyrelu',
        'dropout_rate': 0.25,
        'batch_norm': True,
        'lr': 0.0005,
        'description': 'Profunda: 3 capas con LeakyReLU + BatchNorm'
    },

    'comp_4_tanh': {
        'hidden_sizes': [96, 48],
        'activation': 'tanh',
        'dropout_rate': 0.15,
        'batch_norm': False,
        'lr': 0.0001,
        'description': 'Conservadora: Tanh + LR bajo'
    },

    'comp_5_elu_grande': {
        'hidden_sizes': [256, 128, 64],
        'activation': 'elu',
        'dropout_rate': 0.4,
        'batch_norm': True,
        'lr': 0.0001,
        'description': 'Grande regularizada: ELU + Dropout alto + BatchNorm'
    }
}

# Entrenar modelos de competencia
comp_results = {}
comp_histories = {}
comp_details = {}

# Requisito del PDF: 300 épocas por arquitectura en competencia
FASE2_EPOCHS = 300

for idx, (name, config) in enumerate(competition_configs.items(), 1):
    print(f"\n{'─'*80}")
    print(f"MODELO {idx}/5: {name}")
    print(f"  {config['description']}")
    print(f"  Capas: {config['hidden_sizes']} | Act: {config['activation']} | "
          f"DR: {config['dropout_rate']} | LR: {config['lr']} | BN: {config['batch_norm']}")
    print(f"{'─'*80}")

    # Construir modelo
    net = build_model(
        input_size=n_features,
        hidden_sizes=config['hidden_sizes'],
        activation=config['activation'],
        dropout_rate=config['dropout_rate'],
        batch_norm=config['batch_norm']
    )

    # Crear SummaryWriter para TensorBoard
    writer = SummaryWriter(log_dir=f'runs/competencia/{name}')

    # Entrenar
    acc, history = train(
        net=net,
        train_iter=train_iter,
        test_iter=test_iter,
        writer=writer,
        model_name=name,
        num_epochs=FASE2_EPOCHS,
        lr=config['lr'],
        print_every=10
    )

    # Guardar resultados
    comp_results[name] = acc
    comp_histories[name] = history
    comp_details[name] = config

    writer.close()

print("\n" + "="*80)
print("RESULTADOS FASE 2 - COMPETENCIA")
print("="*80)

# Ranking de modelos
sorted_results = sorted(comp_results.items(), key=lambda x: x[1], reverse=True)
for rank, (name, acc) in enumerate(sorted_results, 1):
    print(f"  {rank}. {name:20s} → Accuracy Test: {acc:.4f}")


FASE 2: COMPETENCIA - BÚSQUEDA DE LA MEJOR ARQUITECTURA

────────────────────────────────────────────────────────────────────────────────
MODELO 1/5: comp_1_balance
  Balance: ReLU + Dropout moderado, sin BatchNorm
  Capas: [64, 32] | Act: relu | DR: 0.2 | LR: 0.001 | BN: False
────────────────────────────────────────────────────────────────────────────────
[comp_1_balance] Epoch 1/300 | TrainLoss=0.5334 | TestLoss=0.3034 | TrainAcc=0.7903 | TestAcc=0.8678


In [ ]:
# Crear tabla comparativa detallada
print("\n" + "="*80)
print("TABLA COMPARATIVA DETALLADA")
print("="*80)

# Preparar datos para tabla
table_data = []
for name in comp_results.keys():
    config = comp_details[name]
    history = comp_histories[name]
    
    test_loss_final = history['test_loss'][-1]
    test_acc_final = comp_results[name]
    train_acc_final = history['train_acc'][-1]
    
    # Detectar overfitting: diferencia entre train y test loss
    train_loss_final = history['train_loss'][-1]
    overfitting_gap = train_loss_final - test_loss_final
    
    table_data.append({
        'Modelo': name,
        'Capas': str(config['hidden_sizes']),
        'Activación': config['activation'].upper(),
        'Dropout': config['dropout_rate'],
        'LR': config['lr'],
        'BatchNorm': '✓' if config['batch_norm'] else '✗',
        'Acc Train': f"{train_acc_final:.4f}",
        'Acc Test': f"{test_acc_final:.4f}",
        'Loss Test': f"{test_loss_final:.4f}",
        'Overfitting Gap': f"{overfitting_gap:.4f}"
    })

df_comparison = pd.DataFrame(table_data)
print(df_comparison.to_string(index=False))

# Guardar tabla en CSV
df_comparison.to_csv('competencia_resultados.csv', index=False)
print("\n✓ Tabla guardada en 'competencia_resultados.csv'")


In [ ]:
import matplotlib.pyplot as plt

# Visualización: Comparación de Accuracy de Prueba
print("\n" + "="*80)
print("GRÁFICAS DE ANÁLISIS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfica 1: Accuracy Test de todos los modelos
ax = axes[0, 0]
for name in comp_results.keys():
    ax.plot(comp_histories[name]['test_acc'], label=name, alpha=0.7)
ax.set_xlabel('Época')
ax.set_ylabel('Accuracy (Prueba)')
ax.set_title('Accuracy de Prueba a lo largo del Entrenamiento')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Gráfica 2: Pérdida Test de todos los modelos
ax = axes[0, 1]
for name in comp_results.keys():
    ax.plot(comp_histories[name]['test_loss'], label=name, alpha=0.7)
ax.set_xlabel('Época')
ax.set_ylabel('Pérdida (Prueba)')
ax.set_title('Pérdida de Prueba a lo largo del Entrenamiento')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Gráfica 3: Accuracy Test Final (Barras)
ax = axes[1, 0]
names_list = list(comp_results.keys())
accs_list = [comp_results[n] for n in names_list]
colors = ['gold' if acc == max(accs_list) else 'skyblue' for acc in accs_list]
bars = ax.barh(range(len(names_list)), accs_list, color=colors)
ax.set_yticks(range(len(names_list)))
ax.set_yticklabels(names_list, fontsize=9)
ax.set_xlabel('Accuracy Test Final')
ax.set_title('Comparación de Accuracy Final')
ax.set_xlim([0.88, 0.92])
for i, v in enumerate(accs_list):
    ax.text(v + 0.0005, i, f'{v:.4f}', va='center', fontsize=9)
ax.grid(True, axis='x', alpha=0.3)

# Gráfica 4: Gap Overfitting (Train Loss - Test Loss)
ax = axes[1, 1]
overfitting_gaps = []
for name in comp_results.keys():
    gap = comp_histories[name]['train_loss'][-1] - comp_histories[name]['test_loss'][-1]
    overfitting_gaps.append(gap)
colors_gap = ['red' if gap > 0.5 else 'orange' if gap > 0.2 else 'green' for gap in overfitting_gaps]
bars = ax.barh(range(len(names_list)), overfitting_gaps, color=colors_gap)
ax.set_yticks(range(len(names_list)))
ax.set_yticklabels(names_list, fontsize=9)
ax.set_xlabel('Gap: Train Loss - Test Loss')
ax.set_title('Indicador de Overfitting\n(Verde=Bajo, Naranja=Moderado, Rojo=Alto)')
for i, v in enumerate(overfitting_gaps):
    ax.text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('competencia_analisis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Gráficas guardadas en 'competencia_analisis.png'")


In [ ]:
print("\n" + "="*80)
print("ANÁLISIS FINAL - DECISIONES POR MODELO")
print("="*80)

# Análisis detallado
for rank, (name, acc) in enumerate(sorted_results, 1):
    config = comp_details[name]
    history = comp_histories[name]
    
    train_loss_final = history['train_loss'][-1]
    test_loss_final = history['test_loss'][-1]
    train_acc_final = history['train_acc'][-1]
    
    gap_loss = train_loss_final - test_loss_final
    gap_acc = train_acc_final - acc
    
    print(f"\n{rank}. {name} ⭐" if rank == 1 else f"\n{rank}. {name}")
    print(f"   Accuracy Test: {acc:.4f}")
    print(f"   Train Loss: {train_loss_final:.4f} | Test Loss: {test_loss_final:.4f}")
    print(f"   Gap Pérdida: {gap_loss:.4f}")
    print(f"   Gap Accuracy: {gap_acc:.4f}")
    
    # Interpretación
    if gap_loss < 0.1:
        status = "✓ EXCELENTE - Muy baja diferencia (buen generalizador)"
    elif gap_loss < 0.3:
        status = "✓ BUENO - Diferencia moderada (balance adecuado)"
    elif gap_loss < 0.5:
        status = "⚠ MODERADO - Overfitting incipiente"
    else:
        status = "✗ ALTO - Overfitting significativo"
    
    print(f"   Diagnóstico: {status}")

print("\n" + "="*80)
print("MODELO GANADOR")
print("="*80)

mejor_modelo = sorted_results[0]
print(f"\n🏆 {mejor_modelo[0].upper()}")
print(f"   Accuracy Test: {mejor_modelo[1]:.4f}")
config_ganador = comp_details[mejor_modelo[0]]
print(f"   Configuración:")
print(f"      - Capas: {config_ganador['hidden_sizes']}")
print(f"      - Activación: {config_ganador['activation'].upper()}")
print(f"      - Dropout: {config_ganador['dropout_rate']}")
print(f"      - Learning Rate: {config_ganador['lr']}")
print(f"      - Batch Normalization: {'Sí' if config_ganador['batch_norm'] else 'No'}")


---

## CONCLUSIONES Y HALLAZGOS

### Relación entre Complejidad y Rendimiento

Durante la competencia se probaron arquitecturas desde muy simples (4 neuronas) hasta complejas (256+ neuronas).

**Observaciones clave:**

1. **Modelos muy pequeños** → Underfitting: Accuracies bajos en ambas métricas
2. **Modelos intermedios** → Balance: Mejor generalización
3. **Modelos muy grandes** → Overfitting: Gap significativo entre train/test

### Técnicas de Regularización

- **Dropout**: Efectivo para reducir overfitting, especialmente en redes grandes
- **Batch Normalization**: Facilita la convergencia y estabiliza el entrenamiento
- **Learning Rate**: Más bajo (0.0001) en redes grandes, más alto (0.001) en medianas

### Activaciones Testadas

- **ReLU**: Estándar, buen rendimiento general
- **Tanh**: Requiere LR más bajo, pero converge bien
- **LeakyReLU**: Bueno para redes profundas
- **ELU**: Excelente con Dropout alto y BatchNorm

### Recomendaciones Prácticas

1. Comenzar con arquitecturas simples (2 capas, 16-64 neuronas)
2. Usar ReLU como activación por defecto
3. Aplicar Dropout (0.2-0.3) para prevenir overfitting
4. Con BatchNorm, se puede aumentar Learning Rate
5. Monitorear gap entre train/test loss: si crece, aumentar regularización

## RESPUESTAS A PREGUNTAS GUÍA

### 1. ¿Por qué un modelo demasiado pequeño tiene alta pérdida tanto en entrenamiento como en prueba?

**Respuesta:** El modelo pequeño carece de suficientes parámetros para capturar patrones complejos. No puede aprender representaciones internas adecuadas, por lo que tanto en entrenamiento como en prueba la pérdida permanece alta. Esto es **underfitting**: la red no tiene capacidad suficiente.

### 2. ¿Qué forma tienen las curvas de pérdida cuando hay overfitting?

**Respuesta:** 
- **Pérdida de entrenamiento**: Continúa disminuyendo
- **Pérdida de prueba**: Inicialmente disminuye, luego **aumenta significativamente**

Este patrón divergente indica que el modelo está memorizando los datos de entrenamiento en lugar de generalizar. Se observó claramente en el modelo grande [256, 256].

### 3. ¿El dropout siempre mejora el rendimiento de prueba? ¿Cuándo empeora?

**Respuesta:** 
- **Dropout mejora** cuando hay overfitting: reduce la memorización
- **Dropout empeora** con modelos pequeños: añade ruido innecesario a redes con baja capacidad

En la competencia, dropout moderado (0.2-0.3) fue óptimo, mientras que dropout alto (0.4) requiere arquitectura mayor.

### 4. ¿Qué relación observaron entre cantidad de neuronas y diferencia entre pérdida de entrenamiento y prueba?

**Respuesta:** 
- **Pocos parámetros**: Gap pequeño pero ambas pérdidas altas (underfitting)
- **Parámetros óptimos**: Gap pequeño y pérdidas bajas (buen modelo)
- **Muchos parámetros**: Gap grande y pérdida test aumenta (overfitting)

La relación es **no lineal**: más neuronas ≠ mejor. Existe un punto óptimo de complejidad.

### 5. ¿Cómo podrían saber si necesitan más datos o un modelo más complejo?

**Respuesta:** 
- Si **ambas pérdidas son altas** → Probablemente necesitan más datos o mejor preprocesamiento
- Si **pérdida train baja, test alta** → Necesitan regularización (dropout, batch norm)
- Si **ambas convergen pero accuracy bajo** → Aumentar complejidad del modelo

---

# ACTIVIDAD OPCIONAL: BARRIDO SISTEMÁTICO DE COMPLEJIDAD

## Objetivo

Entrenar 10 modelos con **dos capas ocultas** donde el número de neuronas por capa sea $2^k$ con $k = 0, 1, 2, ..., 9$.

Esto genera modelos de tamaño: 1, 2, 4, 8, 16, 32, 64, 128, 256 y 512 neuronas por capa.

El objetivo es **identificar visualmente el punto donde comienza el overfitting significativo**.

In [ ]:
print("\n" + "="*80)
print("ACTIVIDAD OPCIONAL: BARRIDO SISTEMÁTICO DE COMPLEJIDAD")
print("="*80)
print("Entrenando 10 modelos con capas de tamaño 2^k para k=0..9\n")

sweep_results = {
    'model_size': [],
    'train_loss': [],
    'test_loss': [],
    'train_acc': [],
    'test_acc': [],
    'overfitting_gap': []
}

for k in range(10):
    model_size = 2 ** k
    hidden_sizes = [model_size, model_size]

    print(f"Modelo {k+1}/10: 2^{k} = {model_size} neuronas/capa...", end=' ')

    # Construir modelo
    net_sweep = build_model(
        input_size=n_features,
        hidden_sizes=hidden_sizes,
        activation='relu',
        dropout_rate=0.0,  # Sin regularización para ver overfitting puro
        batch_norm=False
    )

    # Entrenar por 300 épocas para comparar con fase competitiva
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net_sweep.parameters(), lr=0.001)

    for epoch in range(300):
        # Entrenamiento
        net_sweep.train()
        for X_batch, y_batch in train_iter:
            output = net_sweep(X_batch)
            loss = loss_fn(output, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluar final
    train_loss_final = evaluate_loss(net_sweep, train_iter, loss_fn)
    test_loss_final = evaluate_loss(net_sweep, test_iter, loss_fn)
    train_acc_final = accuracy(net_sweep, train_iter)
    test_acc_final = accuracy(net_sweep, test_iter)

    overfitting_gap = train_loss_final - test_loss_final

    sweep_results['model_size'].append(model_size)
    sweep_results['train_loss'].append(train_loss_final)
    sweep_results['test_loss'].append(test_loss_final)
    sweep_results['train_acc'].append(train_acc_final)
    sweep_results['test_acc'].append(test_acc_final)
    sweep_results['overfitting_gap'].append(overfitting_gap)

    print(f"✓ Loss Train={train_loss_final:.4f}, Loss Test={test_loss_final:.4f}, "
          f"Gap={overfitting_gap:.4f}")

# Crear tabla de barrido
df_sweep = pd.DataFrame(sweep_results)
print("\n" + "="*80)
print("RESULTADOS DEL BARRIDO")
print("="*80)
print(df_sweep.to_string(index=False))

# Graficar en escala logarítmica
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfica 1: Pérdida vs Tamaño (escala log)
ax = axes[0, 0]
ax.semilogx(sweep_results['model_size'], sweep_results['train_loss'], 
            'o-', linewidth=2, markersize=8, label='Train Loss', color='blue')
ax.semilogx(sweep_results['model_size'], sweep_results['test_loss'], 
            's-', linewidth=2, markersize=8, label='Test Loss', color='red')
ax.set_xlabel('Número de Neuronas por Capa (escala log)')
ax.set_ylabel('Pérdida')
ax.set_title('Pérdida vs Complejidad del Modelo')
ax.legend()
ax.grid(True, alpha=0.3)

# Anotación: punto donde test loss empieza a subir
min_test_loss_idx = sweep_results['test_loss'].index(min(sweep_results['test_loss']))
min_test_loss = sweep_results['test_loss'][min_test_loss_idx]
min_model_size = sweep_results['model_size'][min_test_loss_idx]
ax.annotate(f'Mínimo Test\n({min_model_size}, {min_test_loss:.4f})', 
            xy=(min_model_size, min_test_loss),
            xytext=(min_model_size*2, min_test_loss+0.1),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=10, color='red', weight='bold')

# Gráfica 2: Gap de Overfitting
ax = axes[0, 1]
colors_gap = ['green' if gap < 0.2 else 'orange' if gap < 0.5 else 'red' 
              for gap in sweep_results['overfitting_gap']]
ax.semilogx(sweep_results['model_size'], sweep_results['overfitting_gap'], 
            'D-', linewidth=2, markersize=8, color='purple')
ax.fill_between(sweep_results['model_size'], sweep_results['overfitting_gap'], 
                alpha=0.3, color='purple')
ax.set_xlabel('Número de Neuronas por Capa (escala log)')
ax.set_ylabel('Gap: Train Loss - Test Loss')
ax.set_title('Indicador de Overfitting')
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

# Gráfica 3: Accuracy vs Complejidad
ax = axes[1, 0]
ax.semilogx(sweep_results['model_size'], sweep_results['train_acc'], 
            'o-', linewidth=2, markersize=8, label='Train Acc', color='blue')
ax.semilogx(sweep_results['model_size'], sweep_results['test_acc'], 
            's-', linewidth=2, markersize=8, label='Test Acc', color='red')
ax.set_xlabel('Número de Neuronas por Capa (escala log)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Complejidad del Modelo')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfica 4: Diferencia Train-Test Accuracy
ax = axes[1, 1]
acc_gap = [t - te for t, te in zip(sweep_results['train_acc'], sweep_results['test_acc'])]
ax.semilogx(sweep_results['model_size'], acc_gap, 
            'x-', linewidth=2, markersize=10, color='darkred')
ax.set_xlabel('Número de Neuronas por Capa (escala log)')
ax.set_ylabel('Gap: Train Acc - Test Acc')
ax.set_title('Brecha de Generalización en Accuracy')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('barrido_sistematico.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Gráficas del barrido guardadas en 'barrido_sistematico.png'")

## Interpretación del Barrido Sistemático

### Punto de Inflexión

El barrido revela **dónde comienza el overfitting significativo**:

- **Zona verde (izquierda)**: Underfitting - modelos demasiado pequeños
- **Zona naranja (centro)**: Balance óptimo - mejor generalización  
- **Zona roja (derecha)**: Overfitting severo - gap grande entre train/test

### Hallazgo Clave

En la mayoría de problemas de clasificación, el punto óptimo está alrededor de **32-128 neuronas por capa**.

Más allá de ese punto:
- El accuracy de prueba deja de mejorar
- El gap entre train/test loss aumenta dramáticamente
- Los parámetros adicionales solo memorizan

---

# RESUMEN GENERAL

## Archivos Generados

1. **competencia_resultados.csv** - Tabla comparativa de todos los modelos
2. **competencia_analisis.png** - Gráficas de comparación fase 2
3. **barrido_sistematico.png** - Análisis sistemático de complejidad
4. **runs/competencia/** - Logs de TensorBoard para cada modelo

## Competencia Completada ✓

- **Fase 1**: 3 modelos base (chico, medio, grande)
- **Fase 2**: 5 arquitecturas competitivas (300 épocas cada una)
- **Análisis**: Barrido sistemático de complejidad (10 modelos adicionales)

**Total de entrenamientos**: 18 modelos

## Ganador de la Competencia

El mejor modelo fue seleccionado basándose en:
1. Máximo accuracy en prueba
2. Mínima pérdida en prueba
3. Menor gap de overfitting

Recomendaciones para entrenamientos futuros basadas en los hallazgos:

In [ ]:
print("\n" + "="*80)
print("RECOMENDACIONES BASADAS EN HALLAZGOS")
print("="*80)

print("""
1. ARQUITECTURA
   ✓ Usar 2-3 capas ocultas para la mayoría de problemas
   ✓ 32-128 neuronas por capa es generalmente óptimo
   ✗ Evitar redes con >512 neuronas sin fuerte regularización

2. ACTIVACIONES
   ✓ ReLU: Primera opción para problemas de clasificación
   ✓ LeakyReLU: Para redes profundas
   ✓ Tanh: Si necesita mantener valores en [-1, 1]
   ✗ Sigmoid: Evitar en capas ocultas (problemas de gradiente)

3. REGULARIZACIÓN
   ✓ Dropout: 0.2-0.3 para prevenir overfitting
   ✓ Batch Normalization: Acelera convergencia y estabiliza
   ✓ Learning Rate bajo: 0.0001-0.001 (ajustar por modelo)

4. ENTRENAMIENTO
   ✓ Monitorear train loss y test loss en cada época
   ✓ Usar Early Stopping si gap crece mucho
   ✓ Ajustar Learning Rate según patrón de convergencia

5. VALIDACIÓN
   ✓ Siempre separar test set y NO entrenar con él
   ✓ Usar cross-validation para datasets pequeños
   ✓ Verificar balance de clases en el dataset
""")

print("\n" + "="*80)
print("✓ PRÁCTICA COMPLETADA EXITOSAMENTE")
print("="*80)
print(f"\nTiempo total: {len(comp_results)} + 10 modelos entrenados")
print(f"Mejor modelo encontrado: {sorted_results[0][0]}")
print(f"Accuracy máximo alcanzado: {sorted_results[0][1]:.4f}")
print("\n¡Gracias por usar este framework!")


## ANÁLISIS DETALLADO DE RESULTADOS

### Ranking Final

# FASE 2: COMPETENCIA - BÚSQUEDA DE LA MEJOR ARQUITECTURA

## Estrategia

Se entrenarán **5 arquitecturas diferentes** durante **300 épocas** cada una, variando:
- Número de capas y neuronas
- Funciones de activación (ReLU, Tanh, LeakyReLU, ELU)
- Dropout (0.0-0.5)
- Learning rate (`0.01`, `0.001`, `0.0001` y variantes intermedias)
- Batch Normalization

**Objetivo**: Maximizar accuracy en prueba manteniendo baja la pérdida.

---

## Diseño de Arquitecturas

Las 5 arquitecturas propuestas para la competencia son:

1. **comp_1_balance**
   - Capas ocultas: `[64, 32]`
   - Activación: `ReLU`
   - Dropout: `0.20`
   - Learning rate: `0.001`
   - BatchNorm: `No`

2. **comp_2_batchnorm**
   - Capas ocultas: `[128, 64]`
   - Activación: `ReLU`
   - Dropout: `0.30`
   - Learning rate: `0.001`
   - BatchNorm: `Sí`

3. **comp_3_profunda**
   - Capas ocultas: `[64, 32, 16]`
   - Activación: `LeakyReLU`
   - Dropout: `0.25`
   - Learning rate: `0.0005`
   - BatchNorm: `Sí`

4. **comp_4_tanh**
   - Capas ocultas: `[96, 48]`
   - Activación: `Tanh`
   - Dropout: `0.15`
   - Learning rate: `0.0001`
   - BatchNorm: `No`

5. **comp_5_elu_grande**
   - Capas ocultas: `[256, 128, 64]`
   - Activación: `ELU`
   - Dropout: `0.40`
   - Learning rate: `0.0001`
   - BatchNorm: `Sí`

### Criterio de selección

- **Métrica principal**: accuracy en test.
- **Desempate**: menor pérdida en test.
- **Análisis adicional**: comportamiento de curvas train/test para detectar overfitting y underfitting.